
# TT604A / TT624A — 1º Exercício Computacional: **Amostragem**
**Data:** 02/09/2025  

**Aluno:** _preencha aqui_

Este notebook resolve todas as questões (a)–(e) com células separadas, conforme o enunciado.
**Dica:** execute as células na ordem. Os áudios de saída também são salvos em arquivos `.wav` para você comparar em um player externo.


In [ ]:

# === Célula 1 — Imports, leitura do áudio e utilidades ===

import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
import scipy.signal as signal
import IPython.display as ipd
from espectro import espectro
from filtro import filtro 


# Caminho do áudio (ajuste se necessário)
musica = "creed_overcome.wav"

# Leitura do áudio (Fs em Hz, y em array)
Fs, y = wav.read(musica)
print(f"Arquivo: {musica}")
print(f"Fs (Hz): {Fs}, dtype: {y.dtype}, shape: {y.shape}")

# Converter para mono somando canais (se for estéreo) e normalizar para float32
y = y.astype(np.float32)
if y.ndim == 2 and y.shape[1] == 2:
    y_mono = y[:,0] + y[:,1]
else:
    y_mono = y.squeeze()

# Normalização para [-1, 1] se necessário
max_abs = np.max(np.abs(y_mono))
if max_abs > 0:
    y_mono = y_mono / max_abs

dur = len(y_mono) / Fs
print(f"Duração (s): {dur:.2f}, Amostras: {len(y_mono)}")

# Visualização (trecho curto no tempo)
t = np.arange(len(y_mono)) / Fs
plt.figure(figsize=(10,3))
plt.plot(t[:int(0.05*Fs)], y_mono[:int(0.05*Fs)])
plt.title("Trecho do sinal no tempo (50 ms)")
plt.xlabel("Tempo (s)"); plt.ylabel("Amplitude"); plt.grid(True); plt.tight_layout(); plt.show()


## (a) Espectro do sinal de áudio original — discussão do conteúdo espectral

In [ ]:

# === Célula (a) — Espectro do áudio original ===
espectro(y_mono, Fs, titulo="(a) Espectro do áudio original (mono)")
print("Comente no relatório: faixas de frequência com maior energia (vozes, percussão, etc.),"
      " presença de harmônicos, largura de banda aproximada.")


## (b) Subamostragem sem pré-filtragem (M = 6) — espectro e discussão

In [ ]:

# === Célula (b) — Decimação sem anti-aliasing ===
M = 6
y_dec = y_mono[::M]
Fs_dec = Fs / M
print(f"Fs original = {Fs} Hz, após M={M} => Fs_dec = {Fs_dec:.2f} Hz")
print(f"Tamanho antes: {len(y_mono)}, depois: {len(y_dec)}")

espectro(y_dec, Fs_dec, titulo=f"(b) Espectro após subamostragem por M={M} (sem pré-filtragem)")
print("Comente no relatório: observar réplicas espectrais sobrepostas (aliasing)"
      " e comparar largura de banda aparente com a do sinal original.")


## (c) Escuta: original vs subamostrado (sem filtro) — comentários auditivos

In [ ]:

# === Célula (c) — Escuta dos sinais ===
print("▶️ Áudio original")
display(ipd.Audio(y_mono, rate=int(Fs)))

print("▶️ Áudio subamostrado (sem filtro)")
display(ipd.Audio(y_dec, rate=int(Fs_dec)))

# (Opcional) Salvar WAVs para escuta externa
wav.write("saida_original_mono.wav", int(Fs), (y_mono * 32767).astype(np.int16))
wav.write("saida_subamostrado_sem_filtro.wav", int(Fs_dec), (y_dec * 32767).astype(np.int16))
print("Arquivos salvos: saida_original_mono.wav, saida_subamostrado_sem_filtro.wav")


## (d) Pré-filtragem passa-baixas — escolha de ωp, ωr; espectro e escuta

In [ ]:

# === Célula (d) — Anti-aliasing com FIR passa-baixas ===
# Novo Nyquist após decimação: Fs/(2M). Para evitar aliasing, limitar conteúdo < ~Fs/(2M).
f_nyq = Fs/2
f_target = Fs/(2*M)               # Hz
f_p = 0.80 * f_target             # margem na faixa de passagem
f_r = 1.10 * f_target             # início da faixa de rejeição
omega_p = f_p / f_nyq             # normalizado por Nyquist (0..1)
omega_r = f_r / f_nyq

print(f"Escolhas do filtro: f_p = {f_p:.1f} Hz, f_r = {f_r:.1f} Hz")
print(f"Normalizado (Nyquist=1): ω_p = {omega_p:.4f}, ω_r = {omega_r:.4f}")

numtaps = 513  # filtro mais seletivo
h = filtro(omega_p, omega_r, numtaps=numtaps)

# Resposta em frequência do filtro
w, H = signal.freqz(h, worN=4096)
freqs_hz = (w/np.pi) * f_nyq
plt.figure(figsize=(10,4))
plt.plot(freqs_hz, 20*np.log10(np.abs(H)+1e-9))
plt.title(f"Resposta em frequência do FIR (numtaps={numtaps})")
plt.xlabel("Frequência (Hz)"); plt.ylabel("Magnitude (dB)"); plt.grid(True); plt.tight_layout(); plt.show()

# Filtragem (FIR linear-phase => atraso de (L-1)/2 amostras)
y_lp = signal.lfilter(h, [1.0], y_mono)

# Espectro antes/depois do filtro
espectro(y_mono, Fs, titulo="(d) Espectro do sinal original (referência)")
espectro(y_lp,   Fs, titulo="(d) Espectro após pré-filtragem passa-baixas (anti-aliasing)")

print("▶️ Escuta — sinal filtrado (anti-aliasing)")
display(ipd.Audio(y_lp, rate=int(Fs)))

# Salvar WAV do filtrado
wav.write("saida_filtrado.wav", int(Fs), (y_lp/np.max(np.abs(y_lp)+1e-9) * 32767).astype(np.int16))
print("Arquivo salvo: saida_filtrado.wav")


## (e) Subamostragem do sinal filtrado (M = 6) — espectro, comparação e escuta

In [ ]:

# === Célula (e) — Decimação após anti-aliasing ===
y_lp_dec = y_lp[::M]

espectro(y_dec,     Fs_dec, titulo=f"(e) Referência: espectro do subamostrado SEM filtro (M={M})")
espectro(y_lp_dec,  Fs_dec, titulo=f"(e) Espectro do subamostrado APÓS filtro (M={M})")

print("▶️ Escuta — subamostrado SEM filtro (referência)")
display(ipd.Audio(y_dec, rate=int(Fs_dec)))

print("▶️ Escuta — subamostrado APÓS filtro (anti-aliasing)")
display(ipd.Audio(y_lp_dec, rate=int(Fs_dec)))

# Salvar WAVs finais
wav.write("saida_subamostrado_com_filtro.wav", int(Fs_dec), (y_lp_dec/np.max(np.abs(y_lp_dec)+1e-9) * 32767).astype(np.int16))
print("Arquivo salvo: saida_subamostrado_com_filtro.wav")

print("\\nNo relatório escrito, discuta as diferenças espectrais e auditivas entre:")
print(" - Original vs. subamostrado sem filtro;")
print(" - Subamostrado sem filtro vs. subamostrado após filtro;")
print(" - Justifique a escolha de ω_p e ω_r com base em Fs/(2M).")
